In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import torch
from bridgit import Bridgit, Player, Visualizer
from bridgit.schema import Move
from bridgit.ai import BridgitNet, NetWrapper, MCTS
from bridgit.config import BoardConfig, MCTSConfig, NeuralNetConfig
from bridgit import schema
from plotly.subplots import make_subplots

In [ ]:
board_config = BoardConfig(size=4)
mcts_config = MCTSConfig(num_simulations=200000, solve_terminal=True)
net_config = NeuralNetConfig(num_channels=32, num_res_blocks=2)

net = BridgitNet(board_config, net_config)
wrapper = NetWrapper(net)
mcts = MCTS(wrapper, mcts_config)

game = Bridgit(board_config)

g = board_config.grid_size
print(f"Board: {g}x{g}")
print(f"Legal crossings: {len(game.get_available_moves())}")
print(f"Device: {wrapper.device}")

In [ ]:
game = Bridgit(board_config)
game.make_move(schema.Move(row=1, col=1))
game.make_move(schema.Move(row=2, col=2))
game.make_move(schema.Move(row=3, col=3))

# Get raw net prediction
policy, value = mcts._predict(game)

# Mask to only legal crossings
mask = game.to_mask()
masked_policy = policy * mask
masked_policy /= masked_policy.sum()

fig = make_subplots(rows=1, cols=2, subplot_titles=["Raw Policy (all cells)", "Masked Policy (legal moves only)"])

fig.add_trace(Visualizer.visualize_array(policy, colorscale="Blues").data[0], row=1, col=1)
fig.add_trace(Visualizer.visualize_array(masked_policy, colorscale="Reds").data[0], row=1, col=2)

fig.update_layout(width=700, height=350, title=f"Net value estimate: {value:.3f}")
fig.show()

In [ ]:
game = Bridgit(board_config)
game.make_move(schema.Move(row=1, col=1))
game.make_move(schema.Move(row=2, col=2))
game.make_move(schema.Move(row=3, col=3))

# MCTS search
visit_counts = mcts._search(game).visit_counts()
mcts_probs = mcts.get_action_probs(game, temperature=1.0)

fig = make_subplots(rows=1, cols=2, subplot_titles=["Visit Counts", "MCTS Policy (temp=1)"])

fig.add_trace(Visualizer.visualize_array(visit_counts, colorscale="Greens").data[0], row=1, col=1)
fig.add_trace(Visualizer.visualize_array(mcts_probs, colorscale="Reds").data[0], row=1, col=2)

fig.update_layout(width=700, height=350, title="MCTS concentrates visits on promising moves")
fig.show()

In [ ]:
mcts_probs

In [ ]:
board_config = BoardConfig(size=4)
game = Bridgit(board_config)
# game.make_move(schema.Move(row=1, col=1))
# game.make_move(schema.Move(row=1, col=3))
# game.make_move(schema.Move(row=3, col=3))
# game.make_move(schema.Move(row=1, col=5))
# game.make_move(schema.Move(row=5, col=3))
# game.make_move(schema.Move(row=1, col=7))
Visualizer.visualize_game_state(game.state)

In [ ]:
root = mcts._search(game, verbose=True)

In [ ]:
for i in range(1000): 
    mcts.continue_search(root, 200000, verbose=True)

In [ ]:
root.q_value

In [ ]:
def find_game_overs(node, depth=0):
    if node.game.game_over:
        action = f"({node.action.row},{node.action.col})" if node.action else "root"
        winner = node.game.winner.name
        print(f"{'  ' * depth}{action} depth={depth} winner={winner}")
    for child in node.children.values():
        find_game_overs(child, depth + 1)

find_game_overs(root)

In [ ]:
{move: (child.solved_value, child.visit_count, child.q_value) for move, child in root.children.items()}

In [ ]:
print(f"Root visits: {root.visit_count}")

In [ ]:
def max_depth(node, d=0):
    if not node.children:
        return d
    return max(max_depth(c, d+1) for c in node.children.values())

print(f"Max depth: {max_depth(root)}")

In [ ]:
def count_expanded(node):
    count = 1 if node.is_expanded else 0
    for child in node.children.values():
        count += count_expanded(child)
    return count

def count_terminal_visits(node):
    count = node.visit_count if node.game.game_over else 0
    for child in node.children.values():
        count += count_terminal_visits(child)
    return count

exp = count_expanded(root)
term = count_terminal_visits(root)
print(f"Expanded: {exp}, Terminal visits: {term}, Sum: {exp + term}, Root visits: {root.visit_count}")

In [ ]:
def find_deepest(node, d=0):
    if not node.children:
        return node, d
    return max((find_deepest(c, d+1) for c in node.children.values()), key=lambda x: x[1])

leaf, depth = find_deepest(root)
print(f"Depth: {depth}")
print(f"Game over: {leaf.game.game_over}")
print(f"Available moves: {len(leaf.game.get_available_moves())}")
print(f"Is expanded: {leaf.is_expanded}")
print(f"Move count: {leaf.game.move_count}")

In [ ]:
def count_solved(node, depth=0):
    """Print solved nodes in the tree."""
    if node.solved_value is not None:
        player = node.game.current_player.name
        action = f"({node.action.row},{node.action.col})" if node.action else "root"
        terminal = node.game.game_over
        reason = "terminal" if terminal else "inherited"
        print(f"{'  ' * depth}{action} depth={depth} solved={node.solved_value:+.0f} player={player} ({reason})")
    for child in node.children.values():
        count_solved(child, depth + 1)

count_solved(root)

In [ ]:
def find_at_depth(node, target, d=0):
    if d == target:
        return [node]
    result = []
    for c in node.children.values():
        result.extend(find_at_depth(c, target, d+1))
    return result

depth_11 = find_at_depth(root, 11)
print(f"Nodes at depth 11: {len(depth_11)}")
print(f"Solved: {sum(1 for n in depth_11 if n.solved_value is not None)}")
print(f"Expanded: {sum(1 for n in depth_11 if n.is_expanded)}")
for n in depth_11[:5]:
    terminal_children = sum(1 for c in n.children.values() if c.game.game_over)
    print(f"  solved={n.solved_value} expanded={n.is_expanded} children={len(n.children)} terminal_children={terminal_children}")

In [ ]:
for d in range(7,11):
    nodes = find_at_depth(root, d)
    expanded = sum(1 for n in nodes if n.is_expanded)
    solved = sum(1 for n in nodes if n.solved_value is not None)
    game_overs = sum(1 for n in nodes if n.game.game_over)
    print(f"Depth {d}: {len(nodes)} nodes, {expanded} expanded, {solved} solved, {game_overs} game_over")

In [ ]:
print(f"Root solved: {root.solved_value}")
print(f"Root visits: {root.visit_count}")

# How many of root's children are solved?
for move, child in root.children.items():
    print(f"  ({move.row},{move.col}) visits={child.visit_count} q={child.q_value:.3f} solved={child.solved_value}")

In [ ]:
visit_counts = root.visit_counts()
Visualizer.visualize_array(visit_counts, colorscale="Greens")

In [ ]:
probs = mcts.visit_counts_to_probs(root.visit_counts(), 0)

In [ ]:
flat_idx = torch.multinomial(probs.flatten(), 1).item()
row, col = divmod(flat_idx, probs.shape[1])
canonical_move = Move(row=row, col=col)

In [ ]:
canonical_move